# DeePseudopot tool-use test

Narrow test of *one* leg of the pipeline: given the completed Quantum
ESPRESSO DFT output for a single, unperturbed cubic CsPbBr3 structure (see
`deepseudopot_example/README.md`), can the model adapt the existing
`qe_bands_to_ref.py` + `setup_nn_inputs.py` reference scripts to assemble a
DeePseudopot input bundle, then call the new `train_deepseudopot` tool to
actually produce a trained neural-network pseudopotential?

Structure generation, pseudopotential fetching, and the DFT runs themselves
are already done (see `deepseudopot_example/`) -- the model's job starts at
"DFT output on disk" and ends at "trained model produced by DeePseudopot".

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os

# Set BEFORE importing chem_llm.config, which reads this env var once at
# import time (see chem_llm/config.py) to decide the agent's working
# directory. A dedicated sandbox dir keeps this test's log.jsonl / files
# separate from main.ipynb's WORK_DIR (sandbox/test22).
os.environ.setdefault("MATAGENT_WORK_DIR", "sandbox/dpp_agent_test")

os.chdir(os.path.dirname(os.path.abspath("__file__")) or ".")
from chem_llm import config
from dotenv import load_dotenv
load_dotenv()

# HF_HOME must be set before transformers is imported so it picks up the cache dir.
os.environ["HF_HOME"] = config.HF_HOME
from transformers import AutoModelForCausalLM, AutoTokenizer

from chem_llm.agent_core import run_agent, SYSTEM_PROMPT_TEMPLATE
from chem_llm.tools import TOOLS

print("Work dir:", config.WORK_DIR)
print("DeePseudopot example dir:", config.DPP_EXAMPLE_DIR)
print("Registered tools:", [t["name"] for t in TOOLS])

## Load model

Same Qwen model/config as `main.ipynb` (`chem_llm/config.py`).

In [ ]:
if not config.HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is not set. Run `export HF_TOKEN=hf_xxx` before launching this notebook."
    )
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME, token=config.HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
    token=config.HF_TOKEN,
)

## The task

Heavily guided on purpose: the point of this test is whether the model can
*execute* the adaptation of two existing scripts into a working
DeePseudopot input bundle and drive `train_deepseudopot` to a real trained
model -- not whether it can independently rediscover the file formats or
the VBM alignment convention from scratch. Every non-obvious constant
(the -5.25 eV work function, k-path rows 11/21, the 72-band drop) is
explained and justified so the model applies them rather than guessing or
re-deriving them differently.

In [ ]:
COMPOUND = "CsPbBr3"
JOB = "000"  # the one completed DFT job in deepseudopot_example/calculations/000

TASK = f"""
Your job is to take the completed Quantum ESPRESSO DFT output for a single,
unperturbed cubic {COMPOUND} structure and carry it through to a trained
DeePseudopot neural-network pseudopotential. All of the DFT is already done
for you -- your job starts at "DFT output on disk" and ends at "trained
model produced by DeePseudopot".

Everything you need is in deepseudopot_example/ (already in your working
directory). Read deepseudopot_example/README.md first.

Required workflow:

1. Read deepseudopot_example/qe_bands_to_ref.py in full. Note in particular:
   - generate_qe_kpath() returns the 41 fractional (crystal_b) k-points
     along the band path used for calculations/{JOB}/bands.in -- it is
     hardcoded to that exact path, so it applies UNCHANGED here.
   - parse_band_file(filename) reads a QE bands.x-produced .dat file
     (repeating blocks of "kx ky kz" then that k-point's band energies)
     into (kpoints, bands) arrays.
   - compute_kpath_distances(kpoints) turns k-points into a cumulative
     distance along the path -- the x-axis DeePseudopot's
     expBandStruct_0.par expects in its first column.
   - main() ties these together for ONE job: parses
     calculations/{JOB}/{JOB}.bands.dat, drops the first 72 bands
     (`bands[:, 72:]`) to keep the 128 bands DeePseudopot fits to, and
     writes kpoints_0.par + expBandStruct_0.par -- but it does so by
     writing hardcoded filenames INTO THE CURRENT WORKING DIRECTORY. You
     cannot call this script as-is with run_python: run_python always runs
     with YOUR working directory as cwd, and you need the output written
     into a new nn_inputs_g/ directory, not wherever main() happens to run.
   - The 72-dropped / 128-kept band split, and shift.py's hardcoded k-path
     column indices 104 (VBM) / 105 (CBM) in expBandStruct_0.par, come from
     {COMPOUND}'s electron count (176 electrons -> VBM = band 176) and are
     correct for THIS job unchanged, because the composition and
     pseudopotentials are identical to the validated multi-structure
     pipeline -- only the structure itself differs (one unperturbed cubic
     structure instead of a 50-structure perturbed ensemble). Do not
     re-derive or change these numbers.

2. Read deepseudopot_example/setup_nn_inputs.py in full. Note in particular:
   - parse_pw_input(filename) parses a pw.in file's CELL_PARAMETERS and
     ATOMIC_POSITIONS crystal blocks into (scale_in_bohr, normalized_cell,
     atoms).
   - write_system_par(outfile, scale, cell, atoms) writes DeePseudopot's
     system_0.par format from that.
   - The MAIN LOOP (bottom of the file) does, for each completed
     calculations/NNN job: copy nn_template/ -> nn_inputs/inputs_NNN_<suffix>/,
     write system_0.par from that job's pw.in, run qe_bands_to_ref.py as a
     subprocess with cwd=calculations/NNN to produce kpoints_0.par /
     expBandStruct_0.par THERE, copy those two files into the new
     nn_inputs dir, then run nn_template/shift.py (work-function argument
     -5.25) inside the new nn_inputs dir to align the valence-band maximum,
     and -- for suffix=="g" only -- run nn_template/select_rows.py to keep
     only k-path rows 11 and 21 (1-indexed) of kpoints_0.par and
     expBandStruct_0.par, a fast reduced-k-point bundle.
   - You have exactly ONE completed job (calculations/{JOB}), not a loop
     over many -- adapt this logic into a single new script, you do not
     need (and should not try) to reuse the NNN-loop as-is.

3. Write ONE new Python script, build_nn_inputs.py, that assembles a
   complete DeePseudopot input bundle for this one job at nn_inputs_g/ by
   PORTING (not blindly subprocess-calling) the logic from steps 1-2, using
   explicit paths throughout -- no cwd= subprocess tricks, no relying on a
   script's own hardcoded output filenames:
   a. Copy deepseudopot_example/nn_template/ to nn_inputs_g/
      (shutil.copytree).
   b. Parse deepseudopot_example/calculations/{JOB}/pw.in with an adapted
      parse_pw_input, and write nn_inputs_g/system_0.par with
      write_system_par.
   c. Parse deepseudopot_example/calculations/{JOB}/{JOB}.bands.dat with an
      adapted parse_band_file, build the k-path with generate_qe_kpath and
      compute_kpath_distances, and write nn_inputs_g/kpoints_0.par and
      nn_inputs_g/expBandStruct_0.par (dropping the first 72 bands as in
      step 1).
   d. Apply the SAME valence-band-maximum shift shift.py performs (work
      function -5.25 eV, target gap 1.7 eV -- shift.py's ref_gap): load
      nn_inputs_g/expBandStruct_0.par as a numpy array `bs` (column 0 is
      k-distance, so band columns start at 1), compute
      `vbmax = bs[:, 104].max()`, `cbmin = bs[:, 105].min()`,
      `shift = vbmax - (-5.25)`,
      `shift_for_bandgap = 1.7 - (cbmin - vbmax)`; subtract `shift` from
      every band column (1 onward), then add `shift_for_bandgap` to
      columns 105 onward -- exactly shift.py's arithmetic, applied
      in-process instead of via subprocess. This MUST run on the full
      41-row band structure (searching VBM/CBM over the whole k-path, not
      a subset) -- do this step BEFORE step (e). Overwrite
      expBandStruct_0.par with the result.
   e. Apply the SAME row-selection select_rows.py performs: keep only rows
      11 and 21 (1-indexed) of BOTH kpoints_0.par and expBandStruct_0.par,
      overwriting each file with just those two rows -- this is what makes
      the bundle fast to train (Hamiltonian caching is per k-point and NOT
      parallelized within one system, so fewer k-points is the single
      biggest lever on train_deepseudopot's runtime).

   Run this script with run_python. If it errors, fix it and rerun -- do
   not proceed with a broken or partial nn_inputs_g/.

4. Sanity-check nn_inputs_g/ before training: read_file
   nn_inputs_g/kpoints_0.par and nn_inputs_g/expBandStruct_0.par and
   confirm each has exactly 2 rows, and that expBandStruct_0.par's values
   look like real energies in eV (not NaN, not all zero, not wildly
   outside a plausible few-eV range for a semiconductor band structure).
   Also confirm nn_inputs_g/system_0.par's atoms block lists 20 atoms
   matching calculations/{JOB}/pw.in's ATOMIC_POSITIONS (4 Cs + 4 Pb + 12 Br).

5. Call train_deepseudopot with inputs_folder="nn_inputs_g" and
   results_folder="dpp_results". Check `success` explicitly. If it is
   false, read stderr_tail/stdout_tail, diagnose (a malformed file from
   step 3 -- e.g. a column-index or shape mismatch -- is the most likely
   cause), fix build_nn_inputs.py, rerun it, and call train_deepseudopot
   again. Do not call done while success is false.

6. Once train_deepseudopot succeeds, read_file one of the returned
   final_pot_files entries and confirm it contains real numeric
   pseudopotential data (not empty, not NaN).

Do NOT call generate_cif, get_pseudopotential, or run_espresso_workflow --
the structure and all DFT output already exist and must not be
regenerated. Do NOT modify anything under deepseudopot_example/ (treat it
as read-only reference input); write all new files (build_nn_inputs.py,
nn_inputs_g/, dpp_results/) into your own working directory.

Before calling done, you must have:
- A working nn_inputs_g/ bundle you verified in step 4.
- A train_deepseudopot call that returned success: true.
- Read at least one final_pot_*.dat file from dpp_results/ to confirm it
  has real content.

In the done summary, include:
- trained_model_path and the final_pot_files list from the successful
  train_deepseudopot call.
- A one-paragraph outline of how build_nn_inputs.py adapts
  qe_bands_to_ref.py + setup_nn_inputs.py's logic for a single job.
- Any assumptions made and why (e.g. why the -5.25 eV work function and
  k-path rows 11/21 were reused unchanged from the original pipeline
  rather than re-derived).
"""
print(TASK)

## Run

In [ ]:
os.makedirs(config.WORK_DIR, exist_ok=True)
os.chdir(config.WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
final_state = run_agent(
    TASK,
    model,
    tokenizer,
    verbose=True,
    clear_dir=True,
    load_path=config.DPP_EXAMPLE_DIR,
)

print("\nFINAL STATE:\n", json.dumps(final_state.to_dict(), indent=2))

## Inspect the trained model

Quick check independent of the agent's own self-report: look at what's
actually on disk in `dpp_results/`.

In [ ]:
import glob

results_dir = os.path.join(config.WORK_DIR, "dpp_results")
print("Contents of", results_dir)
for p in sorted(glob.glob(os.path.join(results_dir, "*"))):
    print(" ", p)

final_pots = sorted(glob.glob(os.path.join(results_dir, "final_pot_*.dat")))
print(f"\n{len(final_pots)} final_pot_*.dat file(s) -- these + the last epoch_*_PPmodel.pth ARE the trained neural-network pseudopotential.")